# 01 — Bronze Layer: NIBRS Victims & Offenses
**What this notebook does:**
- Reads raw NIBRS Victims CSV from Volume path
- Reads raw NIBRS Offenses CSV from Volume path
- Cleans column names, casts types, adds audit columns
- Writes `bronze_nibrs_victims` and `bronze_nibrs_offenses` as Delta tables


## 1. Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, IntegerType, DoubleType, BooleanType, TimestampType
)
import gc
gc.collect()
print("Imports OK")
print(f"Spark: {spark.version}")


Imports OK
Spark: 4.1.0


## 2. Paths

In [0]:
VICTIMS_CSV   = "/Volumes/workspace/default/raw_data/LAPD_NIBRS_Victims_Dataset.csv"
OFFENSES_CSV  = "/Volumes/workspace/default/raw_data/LAPD_NIBRS_Offenses_Dataset_2024_to_2025.csv"

BRONZE_VICTIMS_TBL  = "bronze_nibrs_victims"
BRONZE_OFFENSES_TBL = "bronze_nibrs_offenses"

print(f"Victims  : {VICTIMS_CSV}")
print(f"Offenses : {OFFENSES_CSV}")


Victims  : /Volumes/workspace/default/raw_data/LAPD_NIBRS_Victims_Dataset.csv
Offenses : /Volumes/workspace/default/raw_data/LAPD_NIBRS_Offenses_Dataset_2024_to_2025.csv


## 3. Bronze — NIBRS Victims

In [0]:
raw_victims = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "false")
         .option("encoding", "UTF-8")
         .csv(VICTIMS_CSV)
)
print(f"Raw rows : {raw_victims.count():,}  |  Cols: {len(raw_victims.columns)}")
raw_victims.printSchema()


Raw rows : 232,660  |  Cols: 16
root
 |-- CaseNo: string (nullable = true)
 |-- UniqueVictimNo: string (nullable = true)
 |-- Date Rptd: string (nullable = true)
 |-- Date OCC: string (nullable = true)
 |-- Time OCC: string (nullable = true)
 |-- AREA: string (nullable = true)
 |-- AREA NAME: string (nullable = true)
 |-- RPT Dist No: string (nullable = true)
 |-- TotalVictimCount: string (nullable = true)
 |-- Vict Age: string (nullable = true)
 |-- Vict Sex: string (nullable = true)
 |-- Vict Descent: string (nullable = true)
 |-- Victim Type: string (nullable = true)
 |-- Victim Shot: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Status Desc: string (nullable = true)



In [0]:
bronze_victims = (
    raw_victims
    .withColumnRenamed("Date Rptd",        "Date_Rptd")
    .withColumnRenamed("Date OCC",         "Date_OCC")
    .withColumnRenamed("Time OCC",         "Time_OCC")
    .withColumnRenamed("AREA NAME",        "AREA_NAME")
    .withColumnRenamed("RPT Dist No",      "RPT_Dist_No")
    .withColumnRenamed("Vict Age",         "Vict_Age")
    .withColumnRenamed("Vict Sex",         "Vict_Sex")
    .withColumnRenamed("Vict Descent",     "Vict_Descent")
    .withColumnRenamed("Victim Type",      "Victim_Type")
    .withColumnRenamed("Victim Shot",      "Victim_Shot")
    .withColumnRenamed("Status Desc",      "Status_Desc")
)

bronze_victims = (
    bronze_victims
    .withColumn("Date_Rptd",
                F.to_timestamp(F.col("Date_Rptd"), "MM/dd/yyyy hh:mm:ss a"))
    .withColumn("Date_OCC",
                F.to_timestamp(F.col("Date_OCC"),  "MM/dd/yyyy hh:mm:ss a"))
    .withColumn("Time_OCC",         F.col("Time_OCC").cast(IntegerType()))
    .withColumn("AREA",             F.col("AREA").cast(IntegerType()))
    .withColumn("RPT_Dist_No",      F.col("RPT_Dist_No").cast(IntegerType()))
    .withColumn("TotalVictimCount", F.col("TotalVictimCount").cast(IntegerType()))
    .withColumn("Vict_Age",
                F.when(F.col("Vict_Age").cast(IntegerType()) > 0,
                       F.col("Vict_Age").cast(IntegerType()))
                 .otherwise(F.lit(None)))
    .withColumn("Victim_Shot",
                F.when(F.upper(F.col("Victim_Shot")) == "YES", F.lit(True))
                 .when(F.upper(F.col("Victim_Shot")) == "NO",  F.lit(False))
                 .otherwise(F.lit(None)).cast(BooleanType()))
    .withColumn("_source",      F.lit("LAPD_NIBRS_Victims_Dataset.csv"))
    .withColumn("_ingested_at", F.current_timestamp())
)

print(f"Bronze victims rows : {bronze_victims.count():,}")
bronze_victims.printSchema()


Bronze victims rows : 232,660
root
 |-- CaseNo: string (nullable = true)
 |-- UniqueVictimNo: string (nullable = true)
 |-- Date_Rptd: timestamp (nullable = true)
 |-- Date_OCC: timestamp (nullable = true)
 |-- Time_OCC: integer (nullable = true)
 |-- AREA: integer (nullable = true)
 |-- AREA_NAME: string (nullable = true)
 |-- RPT_Dist_No: integer (nullable = true)
 |-- TotalVictimCount: integer (nullable = true)
 |-- Vict_Age: integer (nullable = true)
 |-- Vict_Sex: string (nullable = true)
 |-- Vict_Descent: string (nullable = true)
 |-- Victim_Type: string (nullable = true)
 |-- Victim_Shot: boolean (nullable = true)
 |-- Status: string (nullable = true)
 |-- Status_Desc: string (nullable = true)
 |-- _source: string (nullable = false)
 |-- _ingested_at: timestamp (nullable = false)



In [0]:
(
    bronze_victims.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_VICTIMS_TBL)
)
print(f"✓ '{BRONZE_VICTIMS_TBL}' saved.")
display(spark.table(BRONZE_VICTIMS_TBL).limit(5))


✓ 'bronze_nibrs_victims' saved.


CaseNo,UniqueVictimNo,Date_Rptd,Date_OCC,Time_OCC,AREA,AREA_NAME,RPT_Dist_No,TotalVictimCount,Vict_Age,Vict_Sex,Vict_Descent,Victim_Type,Victim_Shot,Status,Status_Desc,_source,_ingested_at
C259037936,C259037936_0,2025-09-12T00:00:00.000Z,2025-01-21T00:00:00.000Z,835,19,Mission,1907,1,null,null,null,Business,false,40,Investigation Continued,LAPD_NIBRS_Victims_Dataset.csv,2026-04-22T19:32:25.716Z
25173355,25173355_0,2025-09-15T00:00:00.000Z,2025-09-15T00:00:00.000Z,750,12,77th Street,1259,1,29,M,Hispanic,Person,false,40,Investigation Continued,LAPD_NIBRS_Victims_Dataset.csv,2026-04-22T19:32:25.716Z
25170788,25170788_0,2025-09-11T00:00:00.000Z,2025-08-15T00:00:00.000Z,1930,3,Southwest,318,1,null,null,null,Business,false,40,Investigation Continued,LAPD_NIBRS_Victims_Dataset.csv,2026-04-22T19:32:25.716Z
C259036481,C259036481_0,2025-09-11T00:00:00.000Z,2025-08-22T00:00:00.000Z,2010,17,Devonshire,1729,1,null,null,null,Business,false,40,Investigation Continued,LAPD_NIBRS_Victims_Dataset.csv,2026-04-22T19:32:25.716Z
C259037784,C259037784_0,2025-09-20T00:00:00.000Z,2025-09-20T00:00:00.000Z,1610,18,Southeast,1836,1,38,F,Hispanic,Person,false,40,Investigation Continued,LAPD_NIBRS_Victims_Dataset.csv,2026-04-22T19:32:25.716Z


## 4. Bronze — NIBRS Offenses

In [0]:
raw_offenses = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "false")
         .option("encoding", "UTF-8")
         .csv(OFFENSES_CSV)
)
print(f"Raw rows : {raw_offenses.count():,}  |  Cols: {len(raw_offenses.columns)}")
raw_offenses.printSchema()


Raw rows : 250,127  |  Cols: 28
root
 |-- CaseNo: string (nullable = true)
 |-- UniqueNIBRNo: string (nullable = true)
 |-- Date Rptd: string (nullable = true)
 |-- Date OCC: string (nullable = true)
 |-- Time OCC: string (nullable = true)
 |-- AREA: string (nullable = true)
 |-- AREA NAME: string (nullable = true)
 |-- RPT Dist No: string (nullable = true)
 |-- TotalOffenseCount: string (nullable = true)
 |-- Group: string (nullable = true)
 |-- NIBR Code: string (nullable = true)
 |-- NIBR Description: string (nullable = true)
 |-- Crime Against: string (nullable = true)
 |-- Premise Cd: string (nullable = true)
 |-- Premise Desc: string (nullable = true)
 |-- Weapon Used Cd: string (nullable = true)
 |-- Weapon Desc: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Status Desc: string (nullable = true)
 |-- TotalVictimCount: string (nullable = true)
 |-- Victim Shot: string (nullable = true)
 |-- Domestic Violence Crime: string (nullable = true)
 |-- Hate Crime: s

In [0]:
bronze_offenses = (
    raw_offenses
    .withColumnRenamed("Date Rptd",               "Date_Rptd")
    .withColumnRenamed("Date OCC",                "Date_OCC")
    .withColumnRenamed("Time OCC",                "Time_OCC")
    .withColumnRenamed("AREA NAME",               "AREA_NAME")
    .withColumnRenamed("RPT Dist No",             "RPT_Dist_No")
    .withColumnRenamed("NIBR Code",               "NIBR_Code")
    .withColumnRenamed("NIBR Description",        "NIBR_Description")
    .withColumnRenamed("Crime Against",           "Crime_Against")
    .withColumnRenamed("Premise Cd",              "Premise_Cd")
    .withColumnRenamed("Premise Desc",            "Premise_Desc")
    .withColumnRenamed("Weapon Used Cd",          "Weapon_Used_Cd")
    .withColumnRenamed("Weapon Desc",             "Weapon_Desc")
    .withColumnRenamed("Status Desc",             "Status_Desc")
    .withColumnRenamed("Victim Shot",             "Victim_Shot")
    .withColumnRenamed("Domestic Violence Crime", "DomesticViolence")
    .withColumnRenamed("Hate Crime",              "HateCrime")
    .withColumnRenamed("Gang-related Crime",      "GangRelated")
    .withColumnRenamed("Transit-related Crime",   "TransitRelated")
    .withColumnRenamed("Homeless-Victim Crime",   "HomelessVictim")
    .withColumnRenamed("Homeless-Suspect Crime",  "HomelessSuspect")
    .withColumnRenamed("Homeless-Arrestee Crime", "HomelessArrestee")
)

def yn_bool(col_name):
    return (
        F.when(F.upper(F.col(col_name)) == "YES", F.lit(True))
         .when(F.upper(F.col(col_name)) == "NO",  F.lit(False))
         .otherwise(F.lit(None)).cast(BooleanType())
    )

flag_cols = ["Victim_Shot","DomesticViolence","HateCrime",
             "GangRelated","TransitRelated","HomelessVictim",
             "HomelessSuspect","HomelessArrestee"]

# Create temp view and select with type casting
bronze_offenses.createOrReplaceTempView("temp_offenses")

bronze_offenses = spark.sql("""
    SELECT 
        CaseNo,
        UniqueNIBRNo,
        try_cast(Date_Rptd as string) as Date_Rptd,
        try_cast(Date_OCC as string) as Date_OCC,
        try_cast(Time_OCC as int) as Time_OCC,
        cast(AREA as string) as AREA,
        AREA_NAME,
        try_cast(RPT_Dist_No as int) as RPT_Dist_No,
        try_cast(TotalOffenseCount as int) as TotalOffenseCount,
        `Group`,
        NIBR_Code,
        NIBR_Description,
        Crime_Against,
        try_cast(Premise_Cd as int) as Premise_Cd,
        Premise_Desc,
        try_cast(Weapon_Used_Cd as int) as Weapon_Used_Cd,
        Weapon_Desc,
        try_cast(Status as int) as Status,
        Status_Desc,
        try_cast(TotalVictimCount as int) as TotalVictimCount,
        Victim_Shot,
        DomesticViolence,
        HateCrime,
        GangRelated,
        TransitRelated,
        HomelessVictim,
        HomelessSuspect,
        HomelessArrestee,
        'LAPD_NIBRS_Offenses_Dataset.csv' as _source,
        current_timestamp() as _ingested_at
    FROM temp_offenses
""")

# Apply boolean conversion to flag columns
for fc in flag_cols:
    bronze_offenses = bronze_offenses.withColumn(fc, yn_bool(fc))

print(f"Bronze offenses rows : {bronze_offenses.count():,}")
bronze_offenses.printSchema()

Bronze offenses rows : 250,127
root
 |-- CaseNo: string (nullable = true)
 |-- UniqueNIBRNo: string (nullable = true)
 |-- Date_Rptd: string (nullable = true)
 |-- Date_OCC: string (nullable = true)
 |-- Time_OCC: integer (nullable = true)
 |-- AREA: string (nullable = true)
 |-- AREA_NAME: string (nullable = true)
 |-- RPT_Dist_No: integer (nullable = true)
 |-- TotalOffenseCount: integer (nullable = true)
 |-- Group: string (nullable = true)
 |-- NIBR_Code: string (nullable = true)
 |-- NIBR_Description: string (nullable = true)
 |-- Crime_Against: string (nullable = true)
 |-- Premise_Cd: integer (nullable = true)
 |-- Premise_Desc: string (nullable = true)
 |-- Weapon_Used_Cd: integer (nullable = true)
 |-- Weapon_Desc: string (nullable = true)
 |-- Status: integer (nullable = true)
 |-- Status_Desc: string (nullable = true)
 |-- TotalVictimCount: integer (nullable = true)
 |-- Victim_Shot: boolean (nullable = true)
 |-- DomesticViolence: boolean (nullable = true)
 |-- HateCrime: b

In [0]:
(
    bronze_offenses.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_OFFENSES_TBL)
)
print(f"✓ '{BRONZE_OFFENSES_TBL}' saved.")
display(spark.table(BRONZE_OFFENSES_TBL).limit(5))


✓ 'bronze_nibrs_offenses' saved.


CaseNo,UniqueNIBRNo,Date_Rptd,Date_OCC,Time_OCC,AREA,AREA_NAME,RPT_Dist_No,TotalOffenseCount,Group,NIBR_Code,NIBR_Description,Crime_Against,Premise_Cd,Premise_Desc,Weapon_Used_Cd,Weapon_Desc,Status,Status_Desc,TotalVictimCount,Victim_Shot,DomesticViolence,HateCrime,GangRelated,TransitRelated,HomelessVictim,HomelessSuspect,HomelessArrestee,_source,_ingested_at
25151345,25151345_23G_0,08/14/2025 12:00:00 AM,08/13/2025 12:00:00 AM,2100,17,Devonshire,1781,1,A,23G,484(A) - PC - F - Grand Theft - Theft Of Motor Vehicle Parts/Accessories - 23G,Property,null,Apartment/Condominium/Townhouse,null,null,40,Investigation Continued,1,false,false,false,false,false,false,false,false,LAPD_NIBRS_Offenses_Dataset.csv,2026-04-22T19:32:48.557Z
25161221,25161221_90D_0,08/28/2025 12:00:00 AM,08/28/2025 12:00:00 AM,2055,17,Devonshire,1753,1,B,90D,23152(A) - VC - M - Dui Alcohol - 90D,Society,22,Street/Parkway,null,null,40,Investigation Continued,0,false,false,false,false,false,false,false,false,LAPD_NIBRS_Offenses_Dataset.csv,2026-04-22T19:32:48.557Z
25236926,25236926_220_1,12/17/2025 12:00:00 AM,12/17/2025 12:00:00 AM,300,21,Topanga,2144,2,A,220,459 - PC - F - Burglary - Residential - 220,Property,null,Single Family Home,null,null,40,Investigation Continued,1,false,false,false,false,false,false,false,false,LAPD_NIBRS_Offenses_Dataset.csv,2026-04-22T19:32:48.557Z
25154680,25154680_13C_0,08/19/2025 12:00:00 AM,07/05/2025 12:00:00 AM,1100,16,Foothill,1663,1,A,13C,422(A) - PC - F - Criminal Threats - 13C,Person,34,Victim's Residence,null,null,10,Cleared by Arrest,1,false,true,false,false,false,false,false,false,LAPD_NIBRS_Offenses_Dataset.csv,2026-04-22T19:32:48.557Z
25114136,25114136_23H_1,06/22/2025 12:00:00 AM,06/22/2025 12:00:00 AM,1200,6,Hollywood,668,2,A,23H,484(A) - PC - M - Petty Theft - All Other Larceny - 23H,Property,24,Transitional Housing/Halfway House,8,"Bodily Force - Personal Weapons (hands, feet, teeth, etc.)",40,Investigation Continued,1,false,true,false,false,false,false,false,false,LAPD_NIBRS_Offenses_Dataset.csv,2026-04-22T19:32:48.557Z


## 5. Bronze Quality Check

In [0]:
def null_report(tbl_name):
    df    = spark.table(tbl_name)
    total = df.count()
    rows  = [(c,
              df.filter(F.col(c).isNull()).count(),
              round(df.filter(F.col(c).isNull()).count() / total * 100, 2))
             for c in df.columns]
    report = spark.createDataFrame(rows, ["column","null_count","null_pct"])
    print(f"\n=== Null report: {tbl_name}  ({total:,} rows) ===")
    display(report.orderBy(F.desc("null_pct")))

null_report(BRONZE_VICTIMS_TBL)
null_report(BRONZE_OFFENSES_TBL)



=== Null report: bronze_nibrs_victims  (232,660 rows) ===


column,null_count,null_pct
Vict_Age,46289,19.9
Vict_Sex,36522,15.7
Vict_Descent,36522,15.7
CaseNo,0,0.0
UniqueVictimNo,0,0.0
Date_Rptd,0,0.0
Date_OCC,0,0.0
Time_OCC,0,0.0
AREA,0,0.0
AREA_NAME,0,0.0



=== Null report: bronze_nibrs_offenses  (250,127 rows) ===


column,null_count,null_pct
Weapon_Used_Cd,177008,70.77
Weapon_Desc,161895,64.73
Premise_Cd,43798,17.51
Group,4736,1.89
Crime_Against,928,0.37
NIBR_Code,14,0.01
CaseNo,0,0.0
UniqueNIBRNo,0,0.0
Date_Rptd,0,0.0
Date_OCC,0,0.0


In [0]:
for t in [BRONZE_VICTIMS_TBL, BRONZE_OFFENSES_TBL]:
    df = spark.table(t)
    print(f"{t:35s}  rows={df.count():>10,}  cols={len(df.columns)}")
print("\nBronze layer complete.")


bronze_nibrs_victims                 rows=   232,660  cols=18
bronze_nibrs_offenses                rows=   250,127  cols=30

Bronze layer complete.
